# v128 — 条件付き Geodesic Hysteresis 境界拡張 (Barb/先端復元) & 併合提案 Split 3-Fold OOF 検証

**学習ゼロ・Colab GPU 約 8〜10 分で全 3-fold (707 観測 / 1,154 注釈レコード) 完結**。

### 狙う誤差モード
1. **Idea 3 — Geodesic Hysteresis 境界拡張 (`hysteresis_expand_mask`)**:
   - 主催者回答 (Topic 753665) で「視認できる barb（棘・毛状突起）も正解マスクに含める」と明言された一方、現行 `prod` は `refined & proposal` で YOLO 1/4 解像度 proto マスクの外側を無条件に切り落としている（`near_miss` `0.3 ≤ IoU ≤ 0.5` が FN 817 件 + FP 588 件）。
   - `proposal` 内のコア領域 (`prob ≥ 0.45`) と**8近傍連結している成分に限り**、`proposal` 外縁 `max_expand_px` まで `prob ≥ high_thr` の画素を連結拡張する（隣接する別フィラメントの混入を幾何的に遮断）。
2. **Idea 4 — 並走フィラメント過併合の分離 (`split_disconnected_merges`)**:
   - YOLO が近接 2 個体を 1 つの bbox に包み、Refiner が隙間を分離できても単一 RLE として出力される過併合（約 250 件）に対し、主成分の主軸から横方向にオフセットした高確信の第 2 連結成分のみを別個体へ分離する（同一チャネル上の途切れは維持）。

### リーク防止の評価設計
- 各 fold `f ∈ {0, 1, 2}` について、厳密 OOF の `runs/yolo_fold_{f}/weights/best.pt` と `mediseg_edgeattnet_refiner_fold{f}_resume.pt` のみを使用。
- `development` (2011–2016, **547 obs**) で最良 arm を選び、独立な `locked_evaluation` (2017–2022, **160 obs**, 58 時間クラスタ) で一発確認する 2 段階判定を同時出力。


In [ ]:
# [1/5] 依存・Drive マウント・設定
%pip install -q "ultralytics==8.4.118" pycocotools
import hashlib, json, math, os, pickle, shutil, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

class CFG:
    GROUPED_FOLDS_SHA256 = "febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e"
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v128")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    PROB_CACHE_DIR = LOCAL_ROOT / "oof_prob_cache"
    MANIFEST = ARTIFACT_ROOT / "v128_hysteresis_split_3fold_manifest.json"

    FOLDS = (0, 1, 2)
    YOLO_IMGSZ = 1280
    PRIMARY_CONF = 0.26
    PREDICT_IOU = 0.55
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    LOW_THR = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    SPARSE_R = 0.77
    BOOTSTRAP_ITERS = 2000

    # 事前固定 7 arms (過剰探索を排したコンパクト設計)
    ARMS = {
        "prod":                 {"high_thr": 1.0,  "max_expand_px": 0,  "split": False},
        "hyst_h72_r8":          {"high_thr": 0.72, "max_expand_px": 8,  "split": False},
        "hyst_h65_r10":         {"high_thr": 0.65, "max_expand_px": 10, "split": False},
        "hyst_h58_r14":         {"high_thr": 0.58, "max_expand_px": 14, "split": False},
        "split_a160_p14":       {"high_thr": 1.0,  "max_expand_px": 0,  "split": True, "min_sec_area": 160, "min_sec_prob": 0.62, "min_perp": 14.0},
        "split_a260_p18":       {"high_thr": 1.0,  "max_expand_px": 0,  "split": True, "min_sec_area": 260, "min_sec_prob": 0.68, "min_perp": 18.0},
        "combo_hyst72_split160":{"high_thr": 0.72, "max_expand_px": 8,  "split": True, "min_sec_area": 160, "min_sec_prob": 0.62, "min_perp": 14.0},
    }

for p in (CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.PROB_CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "GPU ランタイムを選択してください"
print({"gpu": torch.cuda.get_device_name(0), "ultralytics": version("ultralytics")})


In [ ]:
# [2/5] MediSegEdgeAttUNet + Hysteresis / Merge-Split モジュール
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

"""Geodesic hysteresis boundary expansion & parallel-component merge splitting.

Targets two orthogonal residual PQ loss modes identified in 3-fold error
decomposition (RESEARCH_LOG §2.60–2.66) and Organizer Topic 753665:

1. **Near-miss boundary & barb clipping** (817 FN + 588 FP at 0.3 <= IoU <= 0.5):
   Production ``shrink_mask`` clips ``refined & proposal``, cutting off thin
   filament barbs (explicitly included in GT per Topic 753665) and tapered tips
   whenever YOLO's 1/4-res prototype mask under-covers the extremities.
   :func:`hysteresis_expand_mask` keeps the core seed ``(prob >= low_thr) & proposal``
   and allows connected growth up to ``max_expand_px`` outside ``proposal`` ONLY
   where ``prob >= high_thr`` and connected to the core seed, preventing
   neighboring filaments from leaking in.

2. **Multi-filament proposal merges** (``merge_count = 0.217/record``, ~250 cases):
   When YOLO encloses two nearby filaments in a single proposal box and the
   crop refiner cleanly separates them into disconnected high-confidence masks,
   emitting them as a single disconnected RLE scores IoU <= 0.5 against both
   GTs (0 TP, 1 FP, 2 FN). :func:`split_disconnected_merges` separates large
   high-confidence secondary components ONLY when laterally offset from the
   primary component's principal axis (preserving collinear broken bridges of
   a single filament channel).
"""


import cv2
import numpy as np


def hysteresis_expand_mask(
    proposal: np.ndarray,
    probability: np.ndarray,
    box: tuple[int, int, int, int],
    *,
    low_thr: float = 0.45,
    high_thr: float = 0.70,
    max_expand_px: int = 10,
) -> np.ndarray:
    """Grow ``(prob >= low_thr) & proposal`` along connected ``prob >= high_thr`` barbs/tips."""
    x0, y0, x1, y1 = box
    out = np.zeros_like(proposal, dtype=bool)
    prop_crop = proposal[y0:y1, x0:x1].astype(bool)
    core = (probability >= low_thr) & prop_crop
    if not core.any():
        return out
    if max_expand_px <= 0 or high_thr >= 1.0:
        out[y0:y1, x0:x1] = core
        return out

    ksize = 2 * int(max_expand_px) + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    band = cv2.dilate(prop_crop.astype(np.uint8), kernel) != 0
    cand = core | ((probability >= high_thr) & band)
    num_labels, cc = cv2.connectedComponents(cand.astype(np.uint8), connectivity=8)
    if num_labels <= 2:
        out[y0:y1, x0:x1] = cand
        return out
    keep_ids = np.unique(cc[core])
    keep_ids = keep_ids[keep_ids > 0]
    out[y0:y1, x0:x1] = np.isin(cc, keep_ids)
    return out


def _principal_axis(xs: np.ndarray, ys: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (centroid_xy, unit_principal_axis_xy) of pixel coordinates."""
    pts = np.column_stack([xs.astype(np.float64), ys.astype(np.float64)])
    mean = pts.mean(axis=0)
    if len(pts) < 3:
        return mean, np.array([1.0, 0.0], dtype=np.float64)
    centered = pts - mean
    cov = (centered.T @ centered) / max(len(pts) - 1, 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    axis = eigvecs[:, int(np.argmax(eigvals))]
    norm = float(np.linalg.norm(axis))
    return mean, (axis / norm if norm > 1e-12 else np.array([1.0, 0.0], dtype=np.float64))


def split_disconnected_merges(
    mask: np.ndarray,
    probability: np.ndarray | None = None,
    box: tuple[int, int, int, int] | None = None,
    *,
    min_secondary_area: int = 160,
    min_secondary_prob: float = 0.62,
    min_perp_offset_px: float = 14.0,
) -> list[np.ndarray]:
    """Split laterally offset high-confidence disconnected components while keeping collinear fragments."""
    ys_all, xs_all = np.nonzero(mask)
    if len(xs_all) < 2 * min_secondary_area:
        return [mask]

    bx0, bx1 = int(xs_all.min()), int(xs_all.max()) + 1
    by0, by1 = int(ys_all.min()), int(ys_all.max()) + 1
    sub = mask[by0:by1, bx0:bx1].astype(np.uint8)
    num_labels, cc, stats, _ = cv2.connectedComponentsWithStats(sub, connectivity=8)
    if num_labels <= 2:
        return [mask]

    comp_ids = list(range(1, num_labels))
    comp_ids.sort(key=lambda cid: int(stats[cid, cv2.CC_STAT_AREA]), reverse=True)
    if int(stats[comp_ids[1], cv2.CC_STAT_AREA]) < min_secondary_area:
        return [mask]

    primary_id = comp_ids[0]
    py, px = np.nonzero(cc == primary_id)
    mu0, axis0 = _principal_axis(px + bx0, py + by0)

    prob_full = None
    if probability is not None and box is not None:
        px0, py0, px1, py1 = box
        prob_full = np.zeros(mask.shape, dtype=np.float32)
        prob_full[py0:py1, px0:px1] = probability

    split_ids: list[int] = []
    for cid in comp_ids[1:]:
        area = int(stats[cid, cv2.CC_STAT_AREA])
        if area < min_secondary_area:
            continue
        cy, cx = np.nonzero(cc == cid)
        gx, gy = cx + bx0, cy + by0
        if prob_full is not None:
            mean_p = float(prob_full[gy, gx].mean())
            if mean_p < min_secondary_prob:
                continue
        centroid = np.array([float(gx.mean()), float(gy.mean())], dtype=np.float64)
        delta = centroid - mu0
        perp_dist = abs(delta[0] * axis0[1] - delta[1] * axis0[0])
        if perp_dist >= min_perp_offset_px:
            split_ids.append(cid)

    if not split_ids:
        return [mask]

    out_masks: list[np.ndarray] = []
    remaining = sub.astype(bool).copy()
    for cid in split_ids:
        comp_local = cc == cid
        remaining &= ~comp_local
        inst = np.zeros_like(mask, dtype=bool)
        inst[by0:by1, bx0:bx1] = comp_local
        out_masks.append(inst)

    primary_mask = np.zeros_like(mask, dtype=bool)
    primary_mask[by0:by1, bx0:bx1] = remaining
    out_masks.insert(0, primary_mask)
    return out_masks


In [ ]:
# [3/5] COCO・3-fold 契約検証・前処理画像ステージング
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

TRAIN_JSON = next(Path(CFG.DATA_CACHE).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
coco = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for iid, info in images.items():
    file_to_ids[info["file_name"]].append(iid)
train_files = sorted(file_to_ids)

folds_path = CFG.ARTIFACT_ROOT / "grouped_folds.csv"
folds_sha = file_sha256(folds_path)
assert folds_sha == CFG.GROUPED_FOLDS_SHA256, f"folds SHA mismatch: {folds_sha}"
fold_df = pd.read_csv(folds_path)
fold_of_file = dict(zip(fold_df["file_name"], fold_df["fold"].astype(int)))
role_of_file = {n: ("development" if int(n[:4]) <= 2016 else "locked_evaluation") for n in train_files}

for name in tqdm(train_files, desc="前処理画像をローカル SSD へコピー"):
    dst = CFG.PREPROC_DIR / name
    if not dst.exists():
        shutil.copy2(CFG.PREPROC_CACHE / name, dst)

def to_rle(ann, h, w):
    seg = ann["segmentation"]
    if isinstance(seg, list):
        return mask_utils.merge(mask_utils.frPyObjects(seg, h, w))
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, h, w)
    rle = dict(seg)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

gt_rles = {iid: [to_rle(a, int(info["height"]), int(info["width"]))
                 for a in annotations[iid] if not a.get("iscrowd", 0)]
           for iid, info in images.items()}
print({"total_obs": len(train_files),
       "dev_obs": sum(v == "development" for v in role_of_file.values()),
       "locked_obs": sum(v == "locked_evaluation" for v in role_of_file.values())})


In [ ]:
# [4/5] 3-Fold OOF YOLO + EdgeAtt Refiner 推論キャッシュ (uint8 確率マップ保持)
def mask_square_box(mask, context=CFG.CROP_CONTEXT, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(h, w))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

FOLD_WEIGHT_SHAS = {}
OOF_PROPOSALS = {}

for f in CFG.FOLDS:
    yolo_pt = CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{f}" / "weights" / "best.pt"
    ref_pt = CFG.ARTIFACT_ROOT / f"mediseg_edgeattnet_refiner_fold{f}_resume.pt"
    assert yolo_pt.exists() and ref_pt.exists(), (yolo_pt, ref_pt)
    FOLD_WEIGHT_SHAS[f"fold_{f}"] = {"yolo": file_sha256(yolo_pt), "refiner": file_sha256(ref_pt)}

    cache_pkl = CFG.PROB_CACHE_DIR / f"fold{f}_oof_probs.pkl"
    if cache_pkl.exists():
        fold_rows = pickle.loads(cache_pkl.read_bytes())
        for r in fold_rows:
            OOF_PROPOSALS[r["name"]] = r
        print(f"再利用: fold {f} キャッシュ ({len(fold_rows)} obs)")
        continue

    yolo_model = YOLO(str(yolo_pt))
    refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
    ck = torch.load(ref_pt, map_location="cuda", weights_only=False)
    refiner.load_state_dict(ck.get("best_model") or ck["model"], strict=True)
    del ck

    f_files = [n for n in train_files if fold_of_file[n] == f]
    fold_rows = []
    for name in tqdm(f_files, desc=f"fold {f} OOF YOLO+Refiner"):
        img_path = CFG.PREPROC_DIR / name
        res = yolo_model.predict(str(img_path), imgsz=CFG.YOLO_IMGSZ, conf=CFG.PRIMARY_CONF,
                                 iou=CFG.PREDICT_IOU, agnostic_nms=True, retina_masks=True, verbose=False)[0]
        h, w = res.orig_shape
        masks = []
        if res.masks is not None:
            for m in res.masks.data.detach().cpu().numpy():
                if m.shape != (h, w):
                    m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)
                masks.append(m >= 0.5)
        conf = res.boxes.conf.detach().cpu().numpy().astype(np.float32) if res.boxes is not None else np.zeros(0, np.float32)
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        items = []
        with torch.no_grad():
            for i, pm in enumerate(masks):
                if pm.sum() < CFG.MIN_AREA:
                    continue
                x0, y0, x1, y1 = mask_square_box(pm)
                crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
                ten = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
                prob_320 = refiner(ten).sigmoid()[0, 0].cpu().numpy()
                items.append({
                    "conf": float(conf[i]),
                    "raw_rle": mask_utils.encode(np.asfortranarray(pm.astype(np.uint8))),
                    "box": (x0, y0, x1, y1),
                    "prob_320_u8": np.clip(np.rint(prob_320 * 255.0), 0, 255).astype(np.uint8),
                })
        row = {"name": name, "fold": f, "items": items}
        fold_rows.append(row)
        OOF_PROPOSALS[name] = row
    cache_pkl.write_bytes(pickle.dumps(fold_rows, protocol=pickle.HIGHEST_PROTOCOL))
    del yolo_model, refiner; torch.cuda.empty_cache()
print("全 3-fold OOF 推論完了:", len(OOF_PROPOSALS))


In [ ]:
# [5/5] 全 7 Arm 評価 (dev 547 / locked 160 / full 707) + 対応あり Bootstrap 検証
def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped or self_iou[i, j] <= 0:
                continue
            inter = self_iou[i, j] * (areas[i] + areas[j]) / (1.0 + self_iou[i, j])
            smaller = i if areas[i] <= areas[j] else j
            if inter / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership_prod(masks):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    occupied = np.zeros(masks[0].shape, dtype=bool)
    kept = []
    for idx in sorted(drop_contained_indices(masks), key=lambda i: areas[i]):
        excl = masks[idx] & ~occupied
        if excl.sum() < CFG.MIN_AREA:
            continue
        kept.append(excl); occupied |= excl
    return kept

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

def build_five_day_clusters(obs_names):
    parsed = sorted((datetime.strptime(n[:14], "%Y%m%d%H%M%S"), n) for n in obs_names)
    clusters, cur, prev_dt = [], [parsed[0][1]], parsed[0][0]
    for dt, name in parsed[1:]:
        if (dt - prev_dt).total_seconds() > 5 * 86400:
            clusters.append(cur); cur = [name]
        else:
            cur.append(name)
        prev_dt = dt
    clusters.append(cur)
    return clusters

def paired_bootstrap(per_obs_ref, per_obs_arm, obs_names, block_mode="obs", n_boot=2000, seed=3407):
    rng = np.random.default_rng(seed)
    units = [[n] for n in obs_names] if block_mode == "obs" else build_five_day_clusters(obs_names)
    n_u = len(units)
    deltas = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        pick = rng.integers(0, n_u, size=n_u)
        sr, sa = [0.0, 0, 0, 0], [0.0, 0, 0, 0]
        for ui in pick:
            for nm in units[ui]:
                for k in range(4):
                    sr[k] += per_obs_ref[nm][k]
                    sa[k] += per_obs_arm[nm][k]
        dr = sr[1] + 0.5 * sr[2] + 0.5 * sr[3]
        da = sa[1] + 0.5 * sa[2] + 0.5 * sa[3]
        deltas[b] = (sa[0] / da if da > 0 else 0.0) - (sr[0] / dr if dr > 0 else 0.0)
    return {
        "n_units": n_u,
        "mean_delta": float(deltas.mean()),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
    }

def aggregate_subset(per_obs, obs_subset):
    s = [0.0, 0, 0, 0]
    for n in obs_subset:
        for k in range(4):
            s[k] += per_obs[n][k]
    den = s[1] + 0.5 * s[2] + 0.5 * s[3]
    pq = s[0] / den if den > 0 else 0.0
    sq = s[0] / s[1] if s[1] else 0.0
    rq = s[1] / den if den > 0 else 0.0
    return {"pq": float(pq), "sq": float(sq), "rq": float(rq), "tp": int(s[1]), "fp": int(s[2]), "fn": int(s[3])}

dev_files = [n for n in train_files if role_of_file[n] == "development"]
locked_files = [n for n in train_files if role_of_file[n] == "locked_evaluation"]

PER_OBS_BY_ARM = {}
SUMMARY_BY_ARM = {}

for arm_name, params in CFG.ARMS.items():
    per_obs = {}
    n_split_events = 0
    for name in tqdm(train_files, desc=f"採点 {arm_name}"):
        row = OOF_PROPOSALS[name]
        cand_masks = []
        for it in row["items"]:
            pm = mask_utils.decode(it["raw_rle"]) != 0
            x0, y0, x1, y1 = it["box"]
            prob = cv2.resize(it["prob_320_u8"].astype(np.float32) / 255.0,
                              (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
            m = hysteresis_expand_mask(
                pm, prob, (x0, y0, x1, y1),
                low_thr=CFG.LOW_THR,
                high_thr=params["high_thr"],
                max_expand_px=params["max_expand_px"],
            )
            if m.sum() < CFG.MIN_AREA:
                continue
            if params.get("split", False):
                parts = split_disconnected_merges(
                    m, prob, (x0, y0, x1, y1),
                    min_secondary_area=params["min_sec_area"],
                    min_secondary_prob=params["min_sec_prob"],
                    min_perp_offset_px=params["min_perp"],
                )
                if len(parts) > 1:
                    n_split_events += len(parts) - 1
                cand_masks.extend(parts)
            else:
                cand_masks.append(m)
        final_masks = apply_ownership_prod(cand_masks)
        pred_rles = mask_utils.encode(np.asfortranarray(np.stack(final_masks, -1).astype(np.uint8))) if final_masks else []
        obs_stat = [0.0, 0, 0, 0]
        for iid in file_to_ids[name]:
            truth = gt_rles[iid]
            mat = iou_matrix(truth, pred_rles)
            if len(truth) == 0:
                obs_stat[2] += len(pred_rles)
            elif len(pred_rles) == 0:
                obs_stat[3] += len(truth)
            else:
                hits = mat > 0.5
                obs_stat[0] += float(mat[hits].sum())
                obs_stat[1] += int(hits.sum())
                obs_stat[2] += int((hits.sum(axis=0) == 0).sum())
                obs_stat[3] += int((hits.sum(axis=1) == 0).sum())
        per_obs[name] = tuple(obs_stat)

    PER_OBS_BY_ARM[arm_name] = per_obs
    SUMMARY_BY_ARM[arm_name] = {
        "params": params,
        "split_extra_instances": n_split_events,
        "dev_547": aggregate_subset(per_obs, dev_files),
        "locked_160": aggregate_subset(per_obs, locked_files),
        "full_707": aggregate_subset(per_obs, train_files),
        "by_fold": {
            f"fold_{f}": aggregate_subset(per_obs, [n for n in train_files if fold_of_file[n] == f])
            for f in CFG.FOLDS
        },
    }

prod_obs = PER_OBS_BY_ARM["prod"]
for arm_name, entry in SUMMARY_BY_ARM.items():
    if arm_name == "prod":
        continue
    obs_arm = PER_OBS_BY_ARM[arm_name]
    entry["delta_dev_vs_prod"] = entry["dev_547"]["pq"] - SUMMARY_BY_ARM["prod"]["dev_547"]["pq"]
    entry["delta_locked_vs_prod"] = entry["locked_160"]["pq"] - SUMMARY_BY_ARM["prod"]["locked_160"]["pq"]
    entry["delta_full_vs_prod"] = entry["full_707"]["pq"] - SUMMARY_BY_ARM["prod"]["full_707"]["pq"]
    entry["boot_dev_obs"] = paired_bootstrap(prod_obs, obs_arm, dev_files, "obs", CFG.BOOTSTRAP_ITERS)
    entry["boot_locked_5d"] = paired_bootstrap(prod_obs, obs_arm, locked_files, "5d_cluster", CFG.BOOTSTRAP_ITERS)

manifest = {
    "experiment_id": "v128_hysteresis_and_merge_split_3fold",
    "weights_sha256": FOLD_WEIGHT_SHAS,
    "folds_sha256": folds_sha,
    "results": SUMMARY_BY_ARM,
}
CFG.MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== SUMMARY TABLE (dev 547 -> locked 160 -> full 707) ===")
for k, v in SUMMARY_BY_ARM.items():
    d_dev = v.get("delta_dev_vs_prod", 0.0)
    d_lck = v.get("delta_locked_vs_prod", 0.0)
    d_all = v.get("delta_full_vs_prod", 0.0)
    print(f"{k:24s} | dev={v['dev_547']['pq']:.5f} ({d_dev:+.5f}) | "
          f"locked={v['locked_160']['pq']:.5f} ({d_lck:+.5f}) | "
          f"full3f={v['full_707']['pq']:.5f} ({d_all:+.5f}) | splits={v['split_extra_instances']}")
print("Saved manifest:", CFG.MANIFEST)
